In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q8: TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LOW PRESSURE SWITCH (LPS) CAO NHẤT
q8 = spark.sql('''
    SELECT
        date,
        COUNT(*) AS tong_so_ban_ghi,
        SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) AS so_lan_lps_kich_hoat,
        ROUND(
            SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS ty_le_lps_kich_hoat_pct
    FROM sensor
    GROUP BY date
    HAVING SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) > 0
    ORDER BY so_lan_lps_kich_hoat DESC
    LIMIT 10
''')

# [GIỮ NGUYÊN] In kế hoạch thực thi chi tiết cho báo cáo
print("\n========== EXECUTION PLAN ==========")
q8.explain(True)

# [GIỮ NGUYÊN] Chuyển đổi sang Pandas sau khi giải thích xong
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LPS CAO NHẤT ---")
df_q8 = q8.toPandas()

# Phần hiển thị: Giữ nguyên logic của bạn nhưng thêm bẫy kiểm tra hiển thị
if df_q8.empty:
    print("Mẹo: Bảng kết quả trả về 0 dòng. Điều này có nghĩa là trong tập dữ liệu sensor hiện tại, không có bản ghi nào mà LPS có giá trị bằng 1.")
else:
    try:
        display(df_q8)
    except NameError:
        # Ép in trực tiếp ra console nếu hàm display() của môi trường không hoạt động
        print(df_q8.to_string(index=False))